# Paper skewed-workload result

This notebook reads the released `results/drip/skew_curves.csv` table, then replaces DRIP/HNSW variants with the latest complete matching raw runs in the same directory. Run `gzip -dk results/drip/*.csv.gz` once after cloning.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def locate_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'results' / 'drip').is_dir():
            return candidate
    raise RuntimeError('Could not locate results/drip from the current working directory')


REPO_ROOT = locate_repo_root()
RESULTS_DIR = REPO_ROOT / 'results'
if str(RESULTS_DIR) not in sys.path:
    sys.path.insert(0, str(RESULTS_DIR))
from latest_results import load_skew_curves

DATA_DIR = RESULTS_DIR / 'drip'
FIGURE_DIR = REPO_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

DATASET_ORDER = ['MSMARCO', 'SIFT']
METHOD_ORDER = [
    'DRIP',
    'CrackIVF$^*$',
    'Flat',
    'IVF4096',
    'HNSW',
    'Quake',
    'HNSW (probe centroids)',
    'DRIP (w/o adaptive refinement)',
]
METHOD_COLORS = {
    'DRIP': '#ec003d',
    'DRIP (w/o adaptive refinement)': '#ec003d',
    'CrackIVF$^*$': '#008a55',
    'Flat': '#999999',
    'IVF4096': '#5e0fa8',
    'HNSW': '#2666e6',
    'HNSW (probe centroids)': '#4f7fde',
    'Quake': '#d89000',
}


In [ ]:
curves = load_skew_curves(DATA_DIR)

fig, axes = plt.subplots(
    3,
    len(DATASET_ORDER),
    figsize=(10, 10),
    gridspec_kw={'height_ratios': [1, 1, 0.65]},
)
legend_handles = {}

for column, dataset in enumerate(DATASET_ORDER):
    dataset_curves = curves[curves['dataset'] == dataset]
    axes[0, column].set_title(f'{dataset} (skewed)', fontsize=16, pad=10)

    for method in METHOD_ORDER:
        method_curves = dataset_curves[dataset_curves['method'] == method].sort_values('num_queries')
        if method_curves.empty:
            continue

        color = METHOD_COLORS[method]
        linestyle = method_curves['linestyle'].iloc[0]
        x = method_curves['num_queries']

        if method != 'HNSW (probe centroids)':
            line = axes[0, column].plot(
                x,
                method_curves['cumulative_wall_time_s'],
                color=color,
                linestyle=linestyle,
                linewidth=1.8,
                alpha=0.75,
                label=method,
            )[0]
        else:
            line = None

        latency_line = axes[1, column].plot(
            x,
            method_curves['query_latency_ms'],
            color=color,
            linestyle=linestyle,
            linewidth=1.8,
            alpha=0.75,
        )[0]
        legend_handles.setdefault(method, line if line is not None else latency_line)

        graph_curves = method_curves[
            np.isfinite(method_curves['graph_points_count'])
            & (method_curves['graph_points_count'] > 0)
        ]
        if not graph_curves.empty and method != 'HNSW (probe centroids)':
            axes[2, column].plot(
                graph_curves['num_queries'],
                graph_curves['graph_points_count'],
                color=color,
                linestyle=linestyle,
                linewidth=1.8,
                alpha=0.75,
            )

    for row in range(3):
        axes[row, column].set_xscale('log')
        axes[row, column].set_yscale('log')
        axes[row, column].set_xlim(50, 1e6)
        axes[row, column].set_xticks([1e2, 1e3, 1e4, 1e5, 1e6])
        axes[row, column].minorticks_off()
    axes[2, column].set_xlabel('Number of Queries', fontsize=15)

axes[0, 0].set_ylim(9, 2.04e4)
axes[0, 1].set_ylim(2.9, 3.01e3)
axes[0, 0].set_ylabel('Cumulative Wall Time (s)', fontsize=15)
axes[1, 0].set_ylabel('Query Latency (ms)', fontsize=15)
axes[2, 0].set_ylabel('Graph Points Count', fontsize=15)

fig.legend(
    [legend_handles[name] for name in METHOD_ORDER],
    METHOD_ORDER,
    loc='upper center',
    ncol=4,
    fontsize=11,
    frameon=True,
)
fig.tight_layout()
fig.subplots_adjust(top=0.88)

output_path = FIGURE_DIR / 'main_acc_time_skew.pdf'
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Saved {output_path.relative_to(REPO_ROOT)}')
plt.show()
